# WithU Talk — Phase 4: Bystander Behavior Classifier (방어 / 동조 / 방관)

Module **D**. It runs **only after** Modules A/B flag cyberbullying, and classifies how each
*bystander* (not aggressor, not victim) reacts: **방어** (defends the victim), **동조** (joins/eggs on
the aggressor), **방관** (stands by / ignores).

**Why this is not a fine-tune.** You have **67 labeled reactions total** (동조 43, 방어 14, 방관 10)
across 5 events — far too few to train a supervised classifier. The principled approach is an
**LLM few-shot classifier** grounded in your annotation rubric, with your 67 labels used as (a)
in-prompt exemplars and (b) the evaluation gold. **방관 is handled by log rules**, not the LLM,
because "did nothing" is a metadata signal (read-but-silent, left, unrelated talk), not text.

**The critical design point** (from the Jacobs/Rathnayake framing in your doc): the same token —
`ㅋㅋㅋ`, `ㅇㅈ`, `맞아` — is 동조 or 방어 depending on *whose message it answers*. So the classifier
must see the **victim + aggressor + bystander utterances together**; that's why we pass a context
window, not a lone message.

> **Honest limits:** 67 labels from 5 events. Report agreement per class with the tiny support
> counts shown; treat as directional. This module's value is being the only real anchor you have for
> bystander behavior until more events are collected/synthesized.


In [1]:
 %pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 11.3 MB/s  0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [openai]2m6/7 [openai]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /home/yonsei/python313_env/bin/python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# No GPU needed — this calls an LLM API.
# pip install anthropic openpyxl scikit-learn numpy    (or: pip install openai)
import os, re, json, time, unicodedata, numpy as np
from pathlib import Path
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
EXCEL_PATH   = Path("wangtta.xlsx")   # your labeled corpus (resolver handles Korean names)
EXCEL_SHEET  = "sheet1"
OUT_DIR      = Path("./models/phase4_bystander"); OUT_DIR.mkdir(parents=True, exist_ok=True)

LLM_PROVIDER = "openai"
LLM_MODEL    = "gpt-4o-mini"     
# NOTE: Sonnet 5 / Fable 5 have breaking API changes — stick with 4-6 unless you adapt.
# set your key in the environment before running:  export ANTHROPIC_API_KEY=...   (or OPENAI_API_KEY)

CTX_WINDOW   = 6                      # preceding messages passed as bullying context
N_SHOT       = {"방어": 4, "동조": 6, "방관": 4}   # exemplars per class (rest -> eval)
SEED         = 42

## 1. Extract the 67 bystander gold examples (with context + rationale) from the Excel

In [4]:
def resolve_xlsx(path_or_dir):
    p = Path(os.path.expanduser(str(path_or_dir).strip().strip('"').strip("'")))
    if p.is_file(): return p
    search = p if p.is_dir() else p.parent
    if search.is_dir():
        if p.suffix:
            want = unicodedata.normalize("NFC", p.name)
            for f in search.iterdir():
                if unicodedata.normalize("NFC", f.name) == want: return f
        xs = sorted(search.glob("*.xlsx"))
        if len(xs) == 1: return xs[0]
    raise FileNotFoundError(f"Could not resolve {path_or_dir!r}")

import openpyxl
def load_bystander_gold(path, sheet, k=CTX_WINDOW):
    ws = openpyxl.load_workbook(resolve_xlsx(path), read_only=True)[sheet]
    rows = list(ws.iter_rows(min_row=2, values_only=True))
    C = {"speaker": 3, "text": 4, "event": 6, "by": 13, "subtype": 14, "reason": 16}
    def ctx(i, event):
        out, j = [], i - 1
        while j >= 0 and len(out) < k:
            r = rows[j]
            if r[C["event"]] == event and str(r[C["text"]]).strip():
                out.append((str(r[C["speaker"]]), str(r[C["text"]])))
            j -= 1
        return list(reversed(out))
    gold = []
    for i, r in enumerate(rows):
        lab = str(r[C["by"]]).strip() if r[C["by"]] else ""
        if lab in ("동조", "방어", "방관"):
            gold.append({"event": r[C["event"]], "speaker": str(r[C["speaker"]]),
                         "target_text": str(r[C["text"]]), "context": ctx(i, r[C["event"]]),
                         "subtype": str(r[C["subtype"]] or ""), "reason": str(r[C["reason"]] or ""),
                         "label": lab})
    return gold

gold = load_bystander_gold(EXCEL_PATH, EXCEL_SHEET)
print("gold:", len(gold), "| dist:", dict(Counter(g["label"] for g in gold)),
      "| events:", dict(Counter(g["event"] for g in gold)))

gold: 67 | dist: {'동조': 43, '방어': 14, '방관': 10} | events: {'WCB001': 24, 'WCB002': 13, 'WCB003': 11, 'WCB004': 14, 'WCB005': 5}


## 2. Rubric + few-shot split + prompt builder

In [5]:
LABELS = ["방어", "동조", "방관", "비해당"]

RUBRIC = """너는 초등학생 단체 채팅방에서 '사이버불링이 진행 중인 상황'의 주변인(가해자·피해자가 아닌 참여자) 반응을 분류한다.
분류 라벨:
- 방어: 가해 중단 요구, 피해자 지지·편들기, 항의 (예: "그만해", "욕하지마")
- 동조: 가해 발화에 대한 웃음·호응·강화, 함께 조롱·배제 (예: "ㅋㅋㅋ", "ㅇㅈ", "나가라")
- 방관: 상황을 인지했으나 개입 없이 무관한 이야기로 넘어가거나 무반응
- 비해당: 주변인 반응으로 보기 어려움
핵심 규칙: 같은 말이라도 '누구의 발화에 반응하는가'에 따라 라벨이 달라진다.
피해자를 향한 조롱에 웃으면 동조, 가해자를 저지하면 방어."""

def stratified_split(gold, n_shot, seed=SEED):
    rng = np.random.default_rng(seed)
    order = list(range(len(gold))); rng.shuffle(order)
    got, shots, evals = Counter(), [], []
    for i in order:
        lab = gold[i]["label"]
        if got[lab] < n_shot.get(lab, 0): shots.append(gold[i]); got[lab] += 1
        else: evals.append(gold[i])
    return shots, evals

def render_ctx(ex):
    return "\n".join(f"  {s}: {t}" for s, t in ex["context"]) or "  (직전 맥락 없음)"

def build_user_prompt(shots, target):
    parts = ["# 예시"]
    for e in shots:
        parts.append(f"[맥락]\n{render_ctx(e)}\n[대상 발화] {e['speaker']}: {e['target_text']}\n[정답] {e['label']}")
    parts += ["# 분류할 항목",
              f"[맥락]\n{render_ctx(target)}\n[대상 발화] {target['speaker']}: {target['target_text']}",
              '\nJSON만 출력(설명 금지): {"label":"방어|동조|방관|비해당","reason":"한 문장 근거"}']
    return "\n\n".join(parts)

shots, evals = stratified_split(gold, N_SHOT)
print(f"exemplars={len(shots)} {dict(Counter(s['label'] for s in shots))} | "
      f"eval={len(evals)} {dict(Counter(e['label'] for e in evals))}")

exemplars=14 {'동조': 6, '방관': 4, '방어': 4} | eval=53 {'동조': 37, '방어': 10, '방관': 6}


## 3. LLM call + robust JSON parsing

In [6]:
def call_llm(system, user, model=LLM_MODEL, provider=LLM_PROVIDER, max_tokens=300):
    if provider == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()                       # reads ANTHROPIC_API_KEY
        msg = client.messages.create(model=model, max_tokens=max_tokens,
                                     system=system, messages=[{"role": "user", "content": user}])
        return "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    elif provider == "openai":
        from openai import OpenAI
        client = OpenAI()                          # reads OPENAI_API_KEY
        r = client.chat.completions.create(model=model, max_tokens=max_tokens,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        return r.choices[0].message.content
    raise ValueError(provider)

def parse_label(txt):
    m = re.search(r'\{.*\}', txt, re.DOTALL)
    if m:
        try:
            d = json.loads(m.group(0))
            if d.get("label") in LABELS: return d["label"], d.get("reason", "")
        except Exception: pass
    for lab in LABELS:                              # fallback: first label mentioned
        if lab in txt: return lab, "(parsed from text)"
    return "비해당", "(unparseable)"

def classify_bystander(context, speaker, target_text, shots=shots, retries=2):
    ex = {"context": context, "speaker": speaker, "target_text": target_text}
    user = build_user_prompt(shots, ex)
    for _ in range(retries + 1):
        try:
            return parse_label(call_llm(RUBRIC, user))
        except Exception as e:
            last = e; time.sleep(1.5)
    return "비해당", f"(error: {last})"

## 4. 방관 by log rules (deployment-time, not the LLM)

In [7]:
def detect_bystanding_by_logs(read_after_sec=None, responded=False, left_silently=False,
                              switched_topic=False, no_response_threshold=60):
    """방관 is a METADATA signal, applied in the live messenger after CB is detected.
    Returns True if the participant witnessed the bullying but did not engage.
      read_after_sec : seconds the msg was read-but-unanswered (None = unknown)
      responded      : did they post any on-topic reaction?
      left_silently  : left the room without reacting
      switched_topic : posted unrelated content during active bullying
    """
    if responded: return False
    if left_silently: return True
    if switched_topic: return True
    if read_after_sec is not None and read_after_sec >= no_response_threshold: return True
    return False

# example
print(detect_bystanding_by_logs(read_after_sec=120, responded=False))  # True -> 방관

True


## 5. Evaluate on held-out gold (directional — tiny support)

In [8]:
y_true, y_pred, reasons = [], [], []
for i, e in enumerate(evals):
    lab, why = classify_bystander(e["context"], e["speaker"], e["target_text"])
    y_true.append(e["label"]); y_pred.append(lab); reasons.append(why)
    if (i + 1) % 10 == 0: print(f"  {i+1}/{len(evals)}")

print("\n=== Bystander classifier vs gold (held-out) ===")
print(classification_report(y_true, y_pred, labels=["방어", "동조", "방관"], zero_division=0, digits=3))
print("confusion [rows=true, cols=pred] order 방어/동조/방관/비해당:")
print(confusion_matrix(y_true, y_pred, labels=LABELS))

json.dump({"model": LLM_MODEL, "n_shot": N_SHOT, "n_eval": len(evals),
           "note": "67 labels / 5 events — directional; 방관 also via log rules"},
          open(OUT_DIR/"bystander_meta.json", "w"), ensure_ascii=False, indent=2)
print("\nsaved", OUT_DIR/"bystander_meta.json")

  10/53
  20/53
  30/53
  40/53
  50/53

=== Bystander classifier vs gold (held-out) ===
              precision    recall  f1-score   support

          방어      0.462     0.600     0.522        10
          동조      0.962     0.676     0.794        37
          방관      0.250     0.500     0.333         6

   micro avg      0.667     0.642     0.654        53
   macro avg      0.558     0.592     0.550        53
weighted avg      0.787     0.642     0.690        53

confusion [rows=true, cols=pred] order 방어/동조/방관/비해당:
[[ 6  0  4  0]
 [ 6 25  5  1]
 [ 1  1  3  1]
 [ 0  0  0  0]]

saved models/phase4_bystander/bystander_meta.json


## 6. Inference helper (for the ensemble)

In [9]:
def bystander_reaction(recent_context, speaker, message, logs=None):
    """Called AFTER Modules A/B flag cyberbullying.
      recent_context : list of (speaker, text) leading up to the reaction (the bullying window)
      logs           : optional dict for the 방관 log-rule check
    Returns {'behavior','reason','source'}.
    """
    if logs and detect_bystanding_by_logs(**logs):
        return {"behavior": "방관", "reason": "log rule (no engagement)", "source": "logs"}
    lab, why = classify_bystander(recent_context, speaker, message)
    return {"behavior": lab, "reason": why, "source": "llm"}

demo_ctx = [("가해자","13살이 어딜","가해자"), ]  # replace with real window
print(bystander_reaction([("A","야 초딩 나가라"),("B","ㅋㅋㅋ")], "C", "그만해 욕하지마"))

{'behavior': '방어', 'reason': 'C가 욕을 그만하라고 요청하고 있기 때문이다.', 'source': 'llm'}


## 7. Where this leaves the system

**Saved:** `./models/phase4_bystander/` (`bystander_meta.json`) + the `bystander_reaction()` helper.

**Module inventory now:** A = message toxicity (Phase 1), B = window context (Phase 2), D = bystander
(this). Remaining: **C = exclusion/배제** (metadata + social-network analysis — needs the live
messenger's read/response/leave logs, so it's built against the running app, seeded by your WEX events).

**Then the ensemble + API:** combine `p_toxic` + `context_score` + bystander into a `cb_score`, apply
the .75 / .85 gates from the design doc, and serve the whole thing behind FastAPI as the AI 추론 서버.

**Two things to do with this module beyond running it:**
- **Cost:** each classification is one LLM call. For live traffic, gate it behind A/B (only run when CB
  is already flagged) and consider `claude-haiku-4-5-20251001` for volume.
- **Grow the anchor:** every new labeled event you collect makes both the exemplars and the eval
  stronger — this is the module most starved for data, so it benefits most from Phase 3/4 collection.


In [10]:
   from openai import OpenAI
   _oa = OpenAI(timeout=20)
   def call_llm(system, user, model=None, provider=None, max_tokens=None):
       r = _oa.chat.completions.create(model=LLM_MODEL,
           messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
       return r.choices[0].message.content or ""

   print(call_llm(RUBRIC, '다음을 그대로 출력: {"label":"방어","reason":"test"}'))

{"label":"방어","reason":"test"}
